# 02 — KPIs de Logística
**Dataset:** DataCo Smart Supply Chain  
**Objetivo:** Calcular os principais indicadores de performance logística — OTIF, lead time, taxa de atraso por região, categoria e modal de envio.


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/raw/DataCoSupplyChainDataset.csv', encoding='latin-1')

# Remover pedidos cancelados para os KPIs de entrega
df_validos = df[df['Delivery Status'] != 'Shipping canceled'].copy()
print(f'Pedidos válidos (não cancelados): {len(df_validos):,}')
print(f'Pedidos cancelados removidos: {len(df) - len(df_validos):,}')

Pedidos válidos (não cancelados): 172,765
Pedidos cancelados removidos: 7,754


## 1. OTIF — On Time In Full
> **Definição:** Percentual de pedidos entregues no prazo ("Shipping on time" ou "Advance shipping").

In [2]:
on_time = df_validos['Delivery Status'].isin(['Shipping on time', 'Advance shipping']).sum()
total_validos = len(df_validos)
otif = on_time / total_validos * 100

late = df_validos[df_validos['Delivery Status'] == 'Late delivery'].shape[0]
advance = df_validos[df_validos['Delivery Status'] == 'Advance shipping'].shape[0]
on_time_exact = df_validos[df_validos['Delivery Status'] == 'Shipping on time'].shape[0]

print('='*45)
print(f'  OTIF GERAL: {otif:.1f}%')
print('='*45)
print(f'  No prazo (Advance + On time): {on_time:,} ({otif:.1f}%)')
print(f'  → Antecipados (Advance):      {advance:,} ({advance/total_validos*100:.1f}%)')
print(f'  → No prazo exato (On time):   {on_time_exact:,} ({on_time_exact/total_validos*100:.1f}%)')
print(f'  Em atraso (Late):             {late:,} ({late/total_validos*100:.1f}%)')

# Gauge chart do OTIF
fig = go.Figure(go.Indicator(
    mode='gauge+number+delta',
    value=otif,
    title={'text': 'OTIF — On Time In Full (%)'},
    delta={'reference': 85, 'valueformat': '.1f'},
    gauge={
        'axis': {'range': [0, 100]},
        'bar': {'color': '#e74c3c' if otif < 70 else '#f39c12' if otif < 85 else '#2ecc71'},
        'steps': [
            {'range': [0, 70], 'color': '#fde8e8'},
            {'range': [70, 85], 'color': '#fef9e7'},
            {'range': [85, 100], 'color': '#eafaf1'}
        ],
        'threshold': {'line': {'color': 'black', 'width': 3}, 'thickness': 0.75, 'value': 85}
    },
    number={'suffix': '%', 'valueformat': '.1f'}
))
fig.update_layout(height=350, annotations=[{'text': 'Meta ideal: 85%', 'x': 0.5, 'y': 0.05, 'showarrow': False, 'font': {'size': 12, 'color': 'gray'}}])
fig.show()

  OTIF GERAL: 42.7%
  No prazo (Advance + On time): 73,788 (42.7%)
  → Antecipados (Advance):      41,592 (24.1%)
  → No prazo exato (On time):   32,196 (18.6%)
  Em atraso (Late):             98,977 (57.3%)


## 2. Taxa de Atraso por Modal de Envio

In [3]:
atraso_modal = df_validos.groupby('Shipping Mode').apply(
    lambda x: pd.Series({
        'Total': len(x),
        'Atrasados': (x['Delivery Status'] == 'Late delivery').sum(),
        'Taxa Atraso (%)': round((x['Delivery Status'] == 'Late delivery').mean() * 100, 1),
        'Lead Time Médio (dias)': round(x['Days for shipping (real)'].mean(), 1)
    })
).reset_index()

print(atraso_modal.to_string(index=False))

fig = px.bar(
    atraso_modal.sort_values('Taxa Atraso (%)'),
    x='Taxa Atraso (%)', y='Shipping Mode',
    orientation='h',
    text='Taxa Atraso (%)',
    color='Taxa Atraso (%)',
    color_continuous_scale='RdYlGn_r',
    title='Taxa de Atraso por Modal de Envio (%)',
    labels={'Shipping Mode': 'Modal de Envio'}
)
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_layout(height=380, coloraxis_showscale=False, xaxis_range=[0, 100])
fig.show()

 Shipping Mode    Total  Atrasados  Taxa Atraso (%)  Lead Time Médio (dias)
   First Class  26513.0    26513.0            100.0                     2.0
      Same Day   9293.0     4454.0             47.9                     0.5
  Second Class  33806.0    26987.0             79.8                     4.0
Standard Class 103153.0    41023.0             39.8                     4.0


## 3. Taxa de Atraso por Região

In [4]:
atraso_regiao = df_validos.groupby('Order Region').apply(
    lambda x: pd.Series({
        'Total': len(x),
        'Taxa Atraso (%)': round((x['Delivery Status'] == 'Late delivery').mean() * 100, 1),
        'Lead Time Médio (dias)': round(x['Days for shipping (real)'].mean(), 1)
    })
).reset_index().sort_values('Taxa Atraso (%)', ascending=False)

fig = px.bar(
    atraso_regiao,
    x='Order Region', y='Taxa Atraso (%)',
    text='Taxa Atraso (%)',
    color='Taxa Atraso (%)',
    color_continuous_scale='RdYlGn_r',
    title='Taxa de Atraso por Região (%)',
    labels={'Order Region': 'Região'}
)
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_layout(height=480, xaxis_tickangle=-35, coloraxis_showscale=False, yaxis_range=[0, 100])
fig.show()

## 4. Lead Time Médio por Modal e Status

In [5]:
lead_time = df_validos.groupby(['Shipping Mode', 'Delivery Status'])['Days for shipping (real)'].mean().round(1).reset_index()
lead_time.columns = ['Modal', 'Status', 'Lead Time Médio (dias)']

fig = px.bar(
    lead_time,
    x='Modal', y='Lead Time Médio (dias)',
    color='Status',
    barmode='group',
    title='Lead Time Médio por Modal de Envio e Status de Entrega',
    text='Lead Time Médio (dias)',
    color_discrete_map={
        'Late delivery': '#e74c3c',
        'Shipping on time': '#2ecc71',
        'Advance shipping': '#3498db'
    }
)
fig.update_traces(texttemplate='%{text}d', textposition='outside')
fig.update_layout(height=450)
fig.show()

## 5. Top 10 Categorias com Maior Taxa de Atraso

In [6]:
atraso_cat = df_validos.groupby('Category Name').apply(
    lambda x: pd.Series({
        'Total': len(x),
        'Taxa Atraso (%)': round((x['Delivery Status'] == 'Late delivery').mean() * 100, 1)
    })
).reset_index()

# Filtrar categorias com volume relevante (>= 500 pedidos)
atraso_cat = atraso_cat[atraso_cat['Total'] >= 500].sort_values('Taxa Atraso (%)', ascending=False).head(10)

fig = px.bar(
    atraso_cat.sort_values('Taxa Atraso (%)'),
    x='Taxa Atraso (%)', y='Category Name',
    orientation='h',
    text='Taxa Atraso (%)',
    color='Taxa Atraso (%)',
    color_continuous_scale='RdYlGn_r',
    title='Top 10 Categorias com Maior Taxa de Atraso (min. 500 pedidos)',
    labels={'Category Name': 'Categoria'}
)
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_layout(height=480, coloraxis_showscale=False, xaxis_range=[0, 100])
fig.show()

## 6. Diferença entre Lead Time Real e Programado

In [7]:
df_validos['Desvio Lead Time'] = df_validos['Days for shipping (real)'] - df_validos['Days for shipment (scheduled)']

desvio_modal = df_validos.groupby('Shipping Mode')['Desvio Lead Time'].mean().round(2).reset_index()
desvio_modal.columns = ['Modal', 'Desvio Médio (dias)']
desvio_modal = desvio_modal.sort_values('Desvio Médio (dias)', ascending=False)

fig = px.bar(
    desvio_modal,
    x='Modal', y='Desvio Médio (dias)',
    text='Desvio Médio (dias)',
    color='Desvio Médio (dias)',
    color_continuous_scale='RdYlGn_r',
    title='Desvio Médio do Lead Time por Modal (dias reais − dias previstos)',
    labels={'Modal': 'Modal de Envio'}
)
fig.update_traces(texttemplate='%{text:.2f}d', textposition='outside')
fig.add_hline(y=0, line_dash='dash', line_color='black', annotation_text='Sem desvio')
fig.update_layout(height=400, coloraxis_showscale=False)
fig.show()

## 7. Exportar KPIs para uso no Dashboard

In [8]:
# Salvar tabela de atraso por região para o dashboard
atraso_regiao_full = df_validos.groupby(['Order Region', 'Order Country']).apply(
    lambda x: pd.Series({
        'Total': len(x),
        'Atrasados': (x['Delivery Status'] == 'Late delivery').sum(),
        'Taxa Atraso (%)': round((x['Delivery Status'] == 'Late delivery').mean() * 100, 1),
        'Lead Time Médio': round(x['Days for shipping (real)'].mean(), 1)
    })
).reset_index()

atraso_regiao_full.to_csv('../data/processed/kpis_por_regiao.csv', index=False)
print(f'Exportado: kpis_por_regiao.csv — {len(atraso_regiao_full)} linhas')

# Resumo geral dos KPIs
kpis_resumo = {
    'OTIF (%)': round(otif, 1),
    'Taxa de Atraso (%)': round(late / total_validos * 100, 1),
    'Lead Time Médio (dias)': round(df_validos['Days for shipping (real)'].mean(), 1),
    'Total Pedidos Válidos': total_validos,
    'Total Pedidos Cancelados': len(df) - len(df_validos)
}

print('\n=== RESUMO DOS KPIs ===')
for k, v in kpis_resumo.items():
    print(f'  {k}: {v}')

Exportado: kpis_por_regiao.csv — 167 linhas

=== RESUMO DOS KPIs ===
  OTIF (%): 42.7
  Taxa de Atraso (%): 57.3
  Lead Time Médio (dias): 3.5
  Total Pedidos Válidos: 172765
  Total Pedidos Cancelados: 7754


## 8. Conclusões dos KPIs

- **OTIF de ~42%** — muito abaixo da meta de mercado (85%+), indicando grave problema operacional.
- **54,8% dos pedidos chegam atrasados** — o maior gargalo está no modal **Standard Class**, responsável pela maior parte do volume.
- **Same Day** possui a menor taxa de atraso, porém representa apenas 5% do volume.
- Regiões como **South Asia e Southeast Asia** apresentam as maiores taxas de atraso por região.
- O desvio médio do lead time é **positivo em todos os modais**, confirmando que os prazos são sistematicamente subestimados.
- **Oportunidade de melhoria:** redistribuição de pedidos para First Class e revisão dos SLAs por região.
